In [1]:
import torch
from pathlib import Path

# Riproducibilita'
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Path del progetto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"\nProject root: {PROJECT_ROOT}")

Device: cuda
GPU: NVIDIA GeForce RTX 5070
VRAM totale: 12.8 GB

Project root: c:\dev\image-captioning


In [2]:
from transformers import CLIPVisionModel, CLIPImageProcessor

CLIP_NAME = "openai/clip-vit-large-patch14"

print(f"Caricamento {CLIP_NAME}...")
print("(prima volta: scarica ~1.7 GB)")
print()

clip_vision = CLIPVisionModel.from_pretrained(CLIP_NAME)
clip_processor = CLIPImageProcessor.from_pretrained(CLIP_NAME)

# Ispezione configurazione
cfg = clip_vision.config
print(f"=== Configurazione CLIP-L ===")
print(f"  hidden_size:        {cfg.hidden_size}")
print(f"  num_hidden_layers:  {cfg.num_hidden_layers}")
print(f"  num_attention_heads:{cfg.num_attention_heads}")
print(f"  image_size:         {cfg.image_size}")
print(f"  patch_size:         {cfg.patch_size}")
print(f"  Patch totali:       {(cfg.image_size // cfg.patch_size)**2}")
print()

# Parametri
n_params = sum(p.numel() for p in clip_vision.parameters())
print(f"Parametri CLIP vision encoder: {n_params/1e6:.1f}M")
print()

# Confronto con il vecchio encoder ViT-base
from transformers import ViTModel
vit_base = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
n_params_vit = sum(p.numel() for p in vit_base.parameters())
print(f"Parametri ViT-base (vecchio encoder): {n_params_vit/1e6:.1f}M")
print(f"CLIP-L e' {n_params/n_params_vit:.1f}x piu' grande")
print(f"hidden_size CLIP-L: {cfg.hidden_size}, ViT-base: {vit_base.config.hidden_size}")

Caricamento openai/clip-vit-large-patch14...
(prima volta: scarica ~1.7 GB)



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

=== Configurazione CLIP-L ===
  hidden_size:        1024
  num_hidden_layers:  24
  num_attention_heads:16
  image_size:         224
  patch_size:         14
  Patch totali:       256

Parametri CLIP vision encoder: 303.2M

Parametri ViT-base (vecchio encoder): 86.4M
CLIP-L e' 3.5x piu' grande
hidden_size CLIP-L: 1024, ViT-base: 768


In [3]:
import torch
import torch.nn as nn
from transformers import (
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    CLIPVisionModel,
    GPT2LMHeadModel,
    GPT2Tokenizer,
)

# Liberiamo la GPU da cose precedenti
torch.cuda.empty_cache()

CLIP_NAME = "openai/clip-vit-large-patch14"
GPT2_NAME = "gpt2"

print("Costruzione modello CLIP-L + GPT-2 small...")

# Encoder CLIP (gia' caricato nella cella precedente, ma lo ricarichiamo per pulizia)
encoder = CLIPVisionModel.from_pretrained(CLIP_NAME)

# Decoder GPT-2 con cross-attention abilitata
# (e' importante: senza is_decoder=True e add_cross_attention=True
# il modello non sa attendere all'output dell'encoder)
from transformers import GPT2Config
decoder_config = GPT2Config.from_pretrained(GPT2_NAME)
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True
decoder = GPT2LMHeadModel.from_pretrained(GPT2_NAME, config=decoder_config)

# VisionEncoderDecoderModel li mette insieme
config = VisionEncoderDecoderConfig.from_encoder_decoder_configs(
    encoder.config, decoder.config
)
model = VisionEncoderDecoderModel(encoder=encoder, decoder=decoder, config=config)

# Config base (BOS/EOS/PAD come negli scenari precedenti)
tokenizer = GPT2Tokenizer.from_pretrained(GPT2_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Conta parametri
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParametri totali:     {n_total/1e6:.1f}M")
print(f"Parametri trainabili: {n_train/1e6:.1f}M")

# Verifica che dentro il modello sia stata creata la proiezione 1024 -> 768
# (HuggingFace lo fa automaticamente se le dimensioni non matchano)
print(f"\nEncoder hidden_size: {model.config.encoder.hidden_size}")
print(f"Decoder hidden_size: {model.config.decoder.hidden_size}")
if hasattr(model, 'enc_to_dec_proj'):
    print(f"Layer di proiezione enc->dec: SI'")
    print(f"  Parametri proiezione: {sum(p.numel() for p in model.enc_to_dec_proj.parameters())/1e6:.2f}M")
else:
    print(f"Layer di proiezione enc->dec: NO (dimensioni gia' compatibili)")

Costruzione modello CLIP-L + GPT-2 small...


Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['h.0.crossattention.c_attn.bias', 'h.0.crossattention.c_attn.weight', 'h.0.crossattention.c_proj.bias', 'h.0.crossattention.c_proj.weight', 'h.0.crossattention.q_attn.bias', 'h.0.crossattention.q_attn.weight', 'h.0.ln_cross_attn.bias', 'h.0.ln_cross_attn.weight', 'h.1.crossattention.c_attn.bias', 'h.1.crossattention.c_attn.weight', 'h.1.crossattention.c_proj.bias', 'h.1.crossattention.c_proj.weight', 'h.1.crossattention.q_attn.bias', 'h.1.crossattention.q_attn.weight', 'h.1.ln_cross_attn.bias', 'h.1.ln_cross_attn.weight', 'h.10.crossattention.c_attn.bias', 'h.10.crossattention.c_attn.weight', 'h.10.crossattention.c_proj.bias', 'h.10.crossattention.c_proj.weight', 'h.10.crossattention.q_attn.bias', 'h.10.crossattention.q_attn.weight', 'h.10.ln_cross_attn.bias', 'h.10.ln_cross_attn.weight', 'h.11.crossattention.c_attn.bias', 'h.11.crossattention.c_attn.weight', 'h.11.crossat


Parametri totali:     456.8M
Parametri trainabili: 456.8M

Encoder hidden_size: 1024
Decoder hidden_size: 768
Layer di proiezione enc->dec: SI'
  Parametri proiezione: 0.79M


c:\dev\image-captioning\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
import time

# Move su GPU
torch.cuda.empty_cache()
model = model.to(device)

# Misura VRAM solo dopo il move del modello
torch.cuda.synchronize()
mem_model = torch.cuda.memory_allocated() / 1e9
print(f"VRAM occupata dal solo modello: {mem_model:.2f} GB")
print()

# Batch finto: 16 immagini 224x224 + label finte
BATCH_SIZE_TEST = 16
SEQ_LEN = 30

fake_pixel_values = torch.randn(BATCH_SIZE_TEST, 3, 224, 224, device=device)
fake_labels = torch.randint(0, 50000, (BATCH_SIZE_TEST, SEQ_LEN), device=device)

print(f"=== Test forward+backward con batch={BATCH_SIZE_TEST} ===")
print()

# Mixed precision come negli scenari precedenti
use_amp = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_amp else torch.float32
print(f"Mixed precision: {'bfloat16' if use_amp else 'fp32'}")

# Optimizer (per misurare anche stati AdamW)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

model.train()

# Forward
torch.cuda.synchronize()
t0 = time.time()
with torch.amp.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp):
    outputs = model(pixel_values=fake_pixel_values, labels=fake_labels)
    loss = outputs.loss
torch.cuda.synchronize()
t_fwd = time.time() - t0

mem_after_fwd = torch.cuda.memory_allocated() / 1e9
print(f"Forward:  {t_fwd*1000:.0f} ms, VRAM: {mem_after_fwd:.2f} GB")

# Backward
torch.cuda.synchronize()
t0 = time.time()
loss.backward()
torch.cuda.synchronize()
t_bwd = time.time() - t0

mem_after_bwd = torch.cuda.memory_allocated() / 1e9
print(f"Backward: {t_bwd*1000:.0f} ms, VRAM: {mem_after_bwd:.2f} GB")

# Optimizer step (genera gli stati m, v di AdamW: e' qui che salta spesso la VRAM)
torch.cuda.synchronize()
t0 = time.time()
optimizer.step()
torch.cuda.synchronize()
t_opt = time.time() - t0

mem_after_opt = torch.cuda.memory_allocated() / 1e9
mem_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"Optim:    {t_opt*1000:.0f} ms, VRAM: {mem_after_opt:.2f} GB")
print()
print(f"=== PICCO VRAM raggiunto: {mem_peak:.2f} GB su {12.8:.1f} GB disponibili ({mem_peak/12.8*100:.0f}%) ===")
print()
print(f"Tempo totale per batch (fwd+bwd+opt): {(t_fwd+t_bwd+t_opt)*1000:.0f} ms")
print(f"Stima tempo per epoca su Flickr30k ({27971//BATCH_SIZE_TEST} batch): "
      f"{(t_fwd+t_bwd+t_opt) * (27971/BATCH_SIZE_TEST) / 60:.1f} min")

# Pulisci
optimizer.zero_grad()
del optimizer, outputs, loss
torch.cuda.empty_cache()

VRAM occupata dal solo modello: 1.85 GB

=== Test forward+backward con batch=16 ===

Mixed precision: bfloat16
Forward:  563 ms, VRAM: 8.51 GB
Backward: 257 ms, VRAM: 3.82 GB
Optim:    139 ms, VRAM: 7.48 GB

=== PICCO VRAM raggiunto: 9.31 GB su 12.8 GB disponibili (73%) ===

Tempo totale per batch (fwd+bwd+opt): 958 ms
Stima tempo per epoca su Flickr30k (1748 batch): 27.9 min


In [5]:
import json

# Scelte di design per la settimana 3
week3_config = {
    "experiment": "week3_clip_l_gpt2",
    "encoder": "openai/clip-vit-large-patch14",
    "encoder_hidden_size": 1024,
    "encoder_params_M": 303.2,
    "decoder": "gpt2",
    "decoder_hidden_size": 768,
    "projection_layer": "1024->768, ~0.79M params (creata automaticamente da HF)",
    "total_params_M": 456.8,

    # Training
    "dataset": "flickr30k",
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "effective_batch_size": 32,
    "num_epochs": 25,
    "early_stop_patience": 3,
    "learning_rate": 5e-5,
    "warmup_ratio": 1/25,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "label_smoothing": 0.1,
    "max_length": 30,

    # Augmentation (stessa di settimana 2)
    "augmentation": "RandomResizedCrop + RandomHorizontalFlip + ColorJitter",

    # Hardware
    "gpu": "RTX 5070",
    "vram_gb": 12.8,
    "vram_peak_measured_gb": 9.31,
    "vram_usage_pct": 73,
    "mixed_precision": "bfloat16",

    # Stima tempo
    "estimated_time_per_epoch_min": 28,
    "estimated_total_time_h_max": 11,  # 25 epoche
    "estimated_total_time_h_realistic": 4,  # con early stopping

    # Selezione checkpoint
    "checkpoint_selection_metric": "BLEU-4 su subset val (200 img)",
    "val_subset_size": 200,
    "evaluation_test_sets": ["flickr8k_test_1091", "flickr30k_test_1906"],
}

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
config_path = PROJECT_ROOT / "outputs" / "week3_config.json"
with open(config_path, "w") as f:
    json.dump(week3_config, f, indent=2)

print(f"Configurazione settimana 3 salvata in:\n  {config_path}")
print()
print(json.dumps(week3_config, indent=2))

Configurazione settimana 3 salvata in:
  c:\dev\image-captioning\outputs\week3_config.json

{
  "experiment": "week3_clip_l_gpt2",
  "encoder": "openai/clip-vit-large-patch14",
  "encoder_hidden_size": 1024,
  "encoder_params_M": 303.2,
  "decoder": "gpt2",
  "decoder_hidden_size": 768,
  "projection_layer": "1024->768, ~0.79M params (creata automaticamente da HF)",
  "total_params_M": 456.8,
  "dataset": "flickr30k",
  "batch_size": 16,
  "gradient_accumulation_steps": 2,
  "effective_batch_size": 32,
  "num_epochs": 25,
  "early_stop_patience": 3,
  "learning_rate": 5e-05,
  "warmup_ratio": 0.04,
  "weight_decay": 0.01,
  "grad_clip": 1.0,
  "label_smoothing": 0.1,
  "max_length": 30,
  "augmentation": "RandomResizedCrop + RandomHorizontalFlip + ColorJitter",
  "gpu": "RTX 5070",
  "vram_gb": 12.8,
  "vram_peak_measured_gb": 9.31,
  "vram_usage_pct": 73,
  "mixed_precision": "bfloat16",
  "estimated_time_per_epoch_min": 28,
  "estimated_total_time_h_max": 11,
  "estimated_total_time_

In [6]:
import gc

# Cleanup
del model, encoder, decoder, fake_pixel_values, fake_labels
gc.collect()
torch.cuda.empty_cache()

mem_now = torch.cuda.memory_allocated() / 1e9
print(f"VRAM dopo cleanup: {mem_now:.2f} GB (dovrebbe essere ~0)")
print()
print("=" * 60)
print("SETUP SETTIMANA 3 COMPLETATO")
print("=" * 60)
print()
print("Prossimo step: creare 11_training_week3.ipynb")
print()
print("Cosa abbiamo verificato:")
print("  [x] CLIP-L scaricato (1.71 GB nella cache HF)")
print("  [x] hidden_size CLIP-L = 1024, proiezione 1024->768 OK")
print("  [x] Modello assemblato: 456.8M parametri")
print("  [x] VRAM measure su batch 16: 9.31 GB picco (73%)")
print("  [x] Batch 16 + grad accum 2 = effective 32")
print("  [x] Config salvata in outputs/week3_config.json")

VRAM dopo cleanup: 0.02 GB (dovrebbe essere ~0)

SETUP SETTIMANA 3 COMPLETATO

Prossimo step: creare 11_training_week3.ipynb

Cosa abbiamo verificato:
  [x] CLIP-L scaricato (1.71 GB nella cache HF)
  [x] hidden_size CLIP-L = 1024, proiezione 1024->768 OK
  [x] Modello assemblato: 456.8M parametri
  [x] VRAM measure su batch 16: 9.31 GB picco (73%)
  [x] Batch 16 + grad accum 2 = effective 32
  [x] Config salvata in outputs/week3_config.json
